## Imports

In [0]:
import boto3
import os

## Get mode and secrets

In [0]:
# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS key names from secrets scope.
def get_scope_key_names(secrets_scope):
    try:
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        return [key.key for key in aws_keys] if aws_keys else []
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)
aws_key = aws_keys[0] if aws_keys else None
aws_secret_key = get_aws_credentials(secrets_scope, aws_key)

# Create SSM and S3 clients
mode = "shared"
if mode == "shared":
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                              aws_access_key_id=aws_key,
                              aws_secret_access_key=aws_secret_key)
    s3_client = boto3.client('s3', region_name='eu-central-1',
                              aws_access_key_id=aws_key,
                              aws_secret_access_key=aws_secret_key)
else:
    ssm_client = boto3.client('ssm', region_name='eu-central-1')
    s3_client = boto3.client('s3', region_name='eu-central-1')

# Function to access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
    response = ssm_client.get_parameter(Name=ssm_key, WithDecryption=False)
    return response['Parameter']['Value']

# Retrieve environment-specific S3 prefix
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

## Sync all files

In [0]:
local_base_folder = f'/Volumes/{env_prefix}-cdh/landed/customerspecialcases/'
bucket_name = f'{env_prefix}-ec1-cepal-common-zone'
s3_base_prefix = f'onprem-outgoing/InternalReports/CustomerSpecialCases/'

try:
    if not os.path.exists(local_base_folder):
        raise FileNotFoundError(f"Local folder not found: {local_base_folder}")

    for root, dirs, files in os.walk(local_base_folder):
        for filename in files:
            full_local_path = os.path.join(root, filename)

            # Extract the subfolder from the root (EXCEPTION, ARCHIVE)
            subfolder = root.split('/')[5]

            # Extract the date from the root (YYYYMMDD)
            date_part = root.split('/')[6]
            
            # Generate the S3 key based on the date part of the filename
            s3_key = f"{s3_base_prefix}{subfolder}/{date_part}/{filename}"

            with open(full_local_path, 'rb') as f:
                s3_client.upload_fileobj(f, bucket_name, s3_key)
            
            print(f"Uploaded: {full_local_path} → s3://{bucket_name}/{s3_key}")
    
    print(f"All files under 'customerspecialcases/' synced successfully.")
except Exception as e:
    print(f"Sync failed: {e}")